In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from datasets import Dataset
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import bitsandbytes as bnb
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
from peft import LoraConfig, get_peft_model, TaskType
from collections import Counter
from transformers import BitsAndBytesConfig
from collections import Counter
import json

from dotenv import load_dotenv
import os
load_dotenv()
YOUR_HF_TOKEN = os.getenv("YOUR_HF_TOKEN")

/home/bistreamt/Desktop/master/research 3/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
with open("data/supreme-court-data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    labels = [item["label"] for item in data if "label" in item]
    
    label_counts = Counter(labels)
    
    print("Number of samples per label:")
    for label, count in sorted(label_counts.items()):
        print(f"{label:12} : {count:4d} samples")
    
    print(f"\nTotal samples in Supreme Court Data: {len(data)}")

Number of samples per label:
admis        :  396 samples
inadmisibil  :  588 samples
respins      :  416 samples

Total samples in Supreme Court Data: 1400


In [4]:
# Define your mapping
path = "data/supreme-court/raw_data.json"
label_map = {
    "respins": 0,
    "admis": 1,
    "inadmisibil": 2,
    "nesoluționat": 3,
}

def prepare_dataset(json_file_path, num_labels):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    if num_labels == 2:
        data = [item for item in data if item['label'] != 'inadmisibil']

    formatted_data = []
    for entry in data:
        # print(f"Processing entry with label: {entry['sentence']}")  # Debugging line
        # Combine description (1-8) and justification (1-4)
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        just = " ".join([entry.get(f"justification_{i}", "") for i in range(1, 5)]).strip()
        sent = " ".join([entry.get(f"sentence", "")]).strip()
        
        formatted_data.append({
            "desc": f"{desc}", 
            "just": f"{just}", # Description prioritized by order
            "sent": f"{sent}",
            "label": label_map[entry["label"]]
        })
    
    
    return Dataset.from_list(formatted_data)

raw = prepare_dataset(path, num_labels=3)

In [5]:
# Find the shortest entry by text length
shortest_idx = min(range(len(raw)), key=lambda i: len(raw[i]['desc'] + raw[i]['just'] + raw[i]['sent']))
shortest_entry = raw[shortest_idx]
length = len(shortest_entry['desc'] + shortest_entry['just'] + shortest_entry['sent'])
print(f"Shortest entry index: {shortest_idx}, length: {length}")
print(shortest_entry)
shortest_entry

Shortest entry index: 1261, length: 1465
{'desc': '1. La 28 iulie 2025, prin sentința Judecătoriei Criuleni, sediul Central Ghenadie Cheptea a fost recunoscut vinovat de comiterea infracțiunii prevăzute de art. 2641 alin. (3) Cod penal și condamnat la 240 ore de muncă neremunerată în folosul comunității, cu anularea dreptului de a conduce mijloace de transport. Avocatul Valeriu Caușnean a depus recurs și cauza fost expediată la Curtea de Apel Centru la 06 august 2025. 2. La 10 septembrie 2025, inculpatul Ghenadie Cheptea, a solicitat Curții Supreme de Justiție strămutarea cauzei penale de la judecătoria Criuleni la o altă instanță de același grad. În motivarea cererii, inculpatul își exprimă neîncrederea față de doi judecători ai Judecătoriei Criuleni, sediul Central și a indicat că aceștia au avut păreri premature și superficiale la examinarea obiectivă, multilaterală și sub toate aspectele a dosarului. 3. La 22 septembrie 2025, cererea de strămutare a fost repartizată judecătorului S

{'desc': '1. La 28 iulie 2025, prin sentința Judecătoriei Criuleni, sediul Central Ghenadie Cheptea a fost recunoscut vinovat de comiterea infracțiunii prevăzute de art. 2641 alin. (3) Cod penal și condamnat la 240 ore de muncă neremunerată în folosul comunității, cu anularea dreptului de a conduce mijloace de transport. Avocatul Valeriu Caușnean a depus recurs și cauza fost expediată la Curtea de Apel Centru la 06 august 2025. 2. La 10 septembrie 2025, inculpatul Ghenadie Cheptea, a solicitat Curții Supreme de Justiție strămutarea cauzei penale de la judecătoria Criuleni la o altă instanță de același grad. În motivarea cererii, inculpatul își exprimă neîncrederea față de doi judecători ai Judecătoriei Criuleni, sediul Central și a indicat că aceștia au avut păreri premature și superficiale la examinarea obiectivă, multilaterală și sub toate aspectele a dosarului. 3. La 22 septembrie 2025, cererea de strămutare a fost repartizată judecătorului Stella Bleșceaga.',
 'just': '4. La moment

In [6]:
with open("data/regional-court-data.json", 'r', encoding='utf-8') as f:
    data = json.load(f)

if isinstance(data, list):
    labels = [item["label"] for item in data if "label" in item]
    
    label_counts = Counter(labels)
    
    print("Number of samples per label:")
    for label, count in sorted(label_counts.items()):
        print(f"{label:12} : {count:4d} samples")
    
    print(f"\nTotal samples in Regional Court Data: {len(data)}")

Number of samples per label:
nevinovat    : 3626 samples
vinovat      : 5244 samples

Total samples in Regional Court Data: 8870


In [7]:
# Define your mapping
path = "data/regional-court-data.json"
label_map = {
    "vinovat": 0,
    "nevinovat": 1
}

def prepare_dataset(json_file_path):
    with open(json_file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    formatted_data = []
    for entry in data:
        # Combine description (1-8) and justification (1-4)
        desc = " ".join([entry.get(f"case_description_{i}", "") for i in range(1, 9)]).strip()
        sent = " ".join([entry.get(f"sentence", "")]).strip()
        
        formatted_data.append({
            "desc": f"{desc}",
            "sent": f"{sent}", # Sentence prioritized by order
            "label": label_map[entry["label"]]
        })
    
    return Dataset.from_list(formatted_data)

raw = prepare_dataset(path)

In [9]:
# Output the 5 shortest cases with combined desc+sent length >= 1450
candidates = [
    i for i in range(len(raw))
    if len(raw[i]["desc"] + raw[i]["sent"]) >= 1450
]

indices = sorted(candidates, key=lambda i: len(raw[i]["desc"] + raw[i]["sent"]))[:5]

for i in indices:
    entry = raw[i]
    length = len(entry["desc"] + entry["sent"])
    print(f"#{i}  len={length}  label={entry['label']}")
    print("DESC:", entry["desc"])
    print("SENT:", entry["sent"])
    print("-" * 80)

#636  len=1458  label=1
DESC: Prin sentința judecătoriei r­lui Căușeni din11.12.2013, xxxxNUMExxxx a fost condamnat cu stabilirea pedepsei amendă penală in mărime de 450 u/c, adică 9000 lei pentru comiterea infracţiunii prevăzute de art.2641 al.3 CP RM. În demersul înaintat, executorul solicită înlocuirea amenzii penale în sumă de 9000 lei neachitate cu închisoarea motivând cu faptul, că xxxxNUMExxxx se eschivează cu rea voinţă de la achitarea amenzii aplicate. În şedinţa de judecată executorul judecătoresc nu sa prezentat, prin cerere a solicitat examinarea demersului în lipsa ei și respingerea demersului din motivul achitării de către condamnat a amenzii aplicate. xxxxNUMExxxx Ion in judecata nu s­a prezentat,deși a fost legal citat,despre data,ora și locul ședinței. Procurorul în şedinţa de judecată a solicitat respingerea demersului pe motivul achitării amenzii penale. Ascultând pârtile, examinând materialele cauzei, judecata consideră necesar de a respinge demersul din următoarele